# AI LÀ AI - TACVU1: Modern ViT (RMSNorm + 2D RoPE + MHA + SwiGLU FFN)

Kiến trúc Vision Transformer tối tân (LLaMA-style Vision Block) kết hợp: **RMSNorm**, **2D Rotary Position Embedding (RoPE)**, **Multi-Head Attention** và **SwiGLU Gated FFN**. Tự động lưu Checkpoint tốt nhất và áp dụng Test-Time Augmentation (TTA).

## 1. Cấu hình

In [ ]:
import os
import gc
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Đường dẫn dữ liệu trên Kaggle
DATA_ROOT = Path('/kaggle/input/datasets/khoileeptit/ca2olp26/TACVU1/data')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data')

# Giải nén private_test.zip sang thư mục làm việc nếu chưa có
if not Path('private_test/images').exists():
    zip_path = DATA_ROOT / 'private_test.zip'
    if zip_path.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall('.', pwd=b'629436')

# Siêu tham số tinh chỉnh tối ưu cho Modern ViT
EPOCHS = 25              # 25 Epochs đảm bảo mô hình ViT hội tụ hoàn hảo
IMAGE_SIZE = 224         # Độ phân giải 224x224 (lưới 14x14 = 196 patches)
PATCH_SIZE = 16         # Kích thước mỗi patch (16x16 pixels)
BATCH_SIZE = 32          # Batch 32 lý tưởng trên GPU
LEARNING_RATE = 3e-4     # Tốc độ học 3e-4 kết hợp Cosine Annealing
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[cấu hình] DATA_ROOT: {DATA_ROOT}')
print(f'[cấu hình] Thiết bị: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'[cấu hình] Độ phân giải ảnh: {IMAGE_SIZE}x{IMAGE_SIZE} | Patch: {PATCH_SIZE}x{PATCH_SIZE} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}')


## 2. Mô hình Modern ViT (RMSNorm + 2D RoPE + MHA + SwiGLU) & Phép biến đổi ảnh

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Normalization - Chuẩn hóa ổn định gradient và tối ưu bộ nhớ."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm_x * self.weight


def precompute_2d_rope_freqs(height_patches: int, width_patches: int, head_dim: int, theta: float = 10000.0):
    """Tính trước ma trận góc quay 2D Rotary Position Embedding cho lưới patches."""
    dim_h = head_dim // 2
    dim_w = head_dim // 2
    freqs_h = 1.0 / (theta ** (torch.arange(0, dim_h, 2)[: (dim_h // 2)].float() / dim_h))
    freqs_w = 1.0 / (theta ** (torch.arange(0, dim_w, 2)[: (dim_w // 2)].float() / dim_w))
    pos_h = torch.arange(height_patches, dtype=torch.float32)
    pos_w = torch.arange(width_patches, dtype=torch.float32)
    freqs_h = torch.outer(pos_h, freqs_h)
    freqs_w = torch.outer(pos_w, freqs_w)
    freqs_h = freqs_h[:, None, :].repeat(1, width_patches, 1)
    freqs_w = freqs_w[None, :, :].repeat(height_patches, 1, 1)
    freqs_2d = torch.cat([freqs_h, freqs_w], dim=-1).view(-1, head_dim // 2)
    cos = torch.cos(freqs_2d).repeat_interleave(2, dim=-1)
    sin = torch.sin(freqs_2d).repeat_interleave(2, dim=-1)
    return cos, sin


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    cos = cos.unsqueeze(0).unsqueeze(0).to(x.device, dtype=x.dtype)
    sin = sin.unsqueeze(0).unsqueeze(0).to(x.device, dtype=x.dtype)
    return (x * cos) + (rotate_half(x) * sin)


class SwiGLUFFN(nn.Module):
    """SwiGLU Gated Feed-Forward Network chuẩn LLaMA cho độ phi tuyến cao."""
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # Gate
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)  # Up-proj
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)  # Down-proj
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class RoPEMHA(nn.Module):
    """Multi-Head Self-Attention tích hợp 2D RoPE & Scaled Dot-Product Attention."""
    def __init__(self, dim: int, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        assert dim % num_heads == 0, 'dim must be divisible by num_heads'
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """Modern Block: RMSNorm -> RoPE MHA -> RMSNorm -> SwiGLU FFN."""
    def __init__(self, dim: int, num_heads: int = 8, mlp_ratio: float = 2.67, dropout: float = 0.0):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.norm1 = RMSNorm(dim)
        self.attn = RoPEMHA(dim, num_heads=num_heads, dropout=dropout)
        self.norm2 = RMSNorm(dim)
        self.ffn = SwiGLUFFN(dim, hidden_dim, dropout=dropout)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ffn(self.norm2(x))
        return x


class ModernViT(nn.Module):
    """Vision Transformer hiện đại (RMSNorm + 2D RoPE + MHA + SwiGLU)."""
    def __init__(
        self, img_size=IMAGE_SIZE, patch_size=PATCH_SIZE, in_chans=3, num_classes=2,
        dim=256, depth=6, num_heads=8, mlp_ratio=2.67, dropout=0.1
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_h = img_size // patch_size
        self.grid_w = img_size // patch_size
        self.num_patches = self.grid_h * self.grid_w
        self.patch_embed = nn.Conv2d(in_chans, dim, kernel_size=patch_size, stride=patch_size)
        cos, sin = precompute_2d_rope_freqs(self.grid_h, self.grid_w, dim // num_heads)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim=dim, num_heads=num_heads, mlp_ratio=mlp_ratio, dropout=dropout)
            for _ in range(depth)
        ])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)


# Augmentation sạch: không dùng ColorJitter hay RandomErasing để tránh phá hủy artifacts của ảnh AI
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

model = ModernViT(img_size=IMAGE_SIZE, patch_size=PATCH_SIZE, dim=256, depth=6, num_heads=8)
print(f'[mô hình] ModernViT (RMSNorm + RoPE + MHA + SwiGLU): {sum(p.numel() for p in model.parameters()):,} tham số')


## 3. Dataset

In [ ]:
def load_image(path: Path, tf):
    with Image.open(path) as image:
        return tf(ImageOps.exif_transpose(image).convert('RGB'))


class FaceImages(Dataset):
    """Đọc ảnh khuôn mặt; nạp sẵn danh sách đường dẫn để tối ưu tốc độ I/O."""
    def __init__(self, rows, root, tf, labeled=True):
        self.root = Path(root)
        self.tf = tf
        self.labeled = labeled
        if self.labeled:
            self.image_paths = [self.root / 'images' / Path(p).name for p in rows['path']]
            self.labels = rows['label'].astype(int).tolist()
        else:
            self.image_paths = [self.root / 'images' / fn for fn in rows['file_name']]
            self.file_names = rows['file_name'].tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = load_image(self.image_paths[index], self.tf)
        if self.labeled:
            return image, self.labels[index]
        return image, self.file_names[index]


## 4. Nạp dữ liệu với Stratified Validation Split (85% Train / 15% Val)

In [ ]:
manifest = pd.read_csv(DATA_ROOT / 'train' / 'manifest.csv')

# Chia tập Train (1,700 ảnh) và Validation (300 ảnh) theo phân bố đều nhãn 0 và 1
train_df, val_df = train_test_split(
    manifest, test_size=0.15, random_state=SEED, stratify=manifest['label']
)

query = pd.DataFrame({
    'file_name': sorted(p.name for p in Path('private_test/images').iterdir() if p.is_file())
})

train_loader = DataLoader(
    FaceImages(train_df, DATA_ROOT / 'train', train_transforms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    FaceImages(val_df, DATA_ROOT / 'train', val_transforms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(f'[dữ liệu] Train: {len(train_df):,} ảnh | Phân bố: {train_df.label.value_counts().to_dict()}')
print(f'[dữ liệu] Validation: {len(val_df):,} ảnh | Phân bố: {val_df.label.value_counts().to_dict()}')
print(f'[dữ liệu] Private Test: {len(query):,} ảnh cần dự đoán')


## 5. Huấn luyện & Tự động lưu Checkpoint tốt nhất (Best Val Model)

In [ ]:
model = ModernViT(img_size=IMAGE_SIZE, patch_size=PATCH_SIZE, dim=256, depth=6, num_heads=8).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = 0

print('================== BẮT ĐẦU HUẤN LUYỆN MODERN ViT (RMSNorm + RoPE + MHA + SwiGLU) ==================')

for epoch in range(1, EPOCHS + 1):
    # 1. Huấn luyện (Training)
    model.train()
    train_loss, train_correct, train_seen = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)

    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_seen += labels.size(0)

        pbar.set_postfix({
            'train_loss': f'{train_loss / train_seen:.4f}',
            'train_acc': f'{train_correct / train_seen:.4f}'
        })

    # 2. Đánh giá (Validation)
    model.eval()
    val_loss, val_correct, val_seen = 0.0, 0, 0
    with torch.inference_mode():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)

            val_loss += loss.item() * labels.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_seen += labels.size(0)

    val_acc = val_correct / val_seen
    val_avg_loss = val_loss / val_seen
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    # Tự động lưu checkpoint tốt nhất
    is_best = False
    if (val_acc > best_val_acc) or (val_acc == best_val_acc and val_avg_loss < best_val_loss):
        best_val_acc = val_acc
        best_val_loss = val_avg_loss
        best_epoch = epoch
        torch.save(model.state_dict(), 'best_model.pth')
        is_best = True

    status_tag = f'⭐ [BEST SAVED -> Acc: {best_val_acc:.4f}]' if is_best else ''
    print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss/train_seen:.4f} Acc: {train_correct/train_seen:.4f} | '
          f'Val Loss: {val_avg_loss:.4f} Acc: {val_acc:.4f} | LR: {lr:.6f} {status_tag}')

print(f'\n[huấn luyện] Hoàn tất! Best Model được lưu tại Epoch {best_epoch} với Val Accuracy = {best_val_acc:.4f}')


## 6. Dự đoán bằng Best Checkpoint + Test-Time Augmentation (TTA)

In [ ]:
started = time.time()

# Nạp lại trọng số tốt nhất đã lưu
model = ModernViT(img_size=IMAGE_SIZE, patch_size=PATCH_SIZE, dim=256, depth=6, num_heads=8)
if Path('best_model.pth').exists():
    model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

test_loader = DataLoader(
    FaceImages(query, Path('private_test'), val_transforms, labeled=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

rows = []

with torch.inference_mode():
    for images, names in tqdm(test_loader, desc='Dự đoán TTA'):
        images = images.to(DEVICE)
        images_flip = torch.flip(images, dims=[-1])  # TTA lật ngang ảnh

        prob_orig = torch.softmax(model(images), dim=1)
        prob_flip = torch.softmax(model(images_flip), dim=1)
        prob_avg = (prob_orig + prob_flip) / 2.0
        labels = prob_avg.argmax(dim=1).cpu().tolist()
        rows.extend(zip(names, labels))

submission = pd.DataFrame(rows, columns=['file_name', 'category_id'])
submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv')

print(f'[dự đoán] Hoàn tất {len(submission):,} ảnh trong {time.time() - started:.1f}s')
print(f'[dự đoán] Phân bố nhãn dự đoán: {submission.category_id.value_counts().to_dict()}')
print('[nộp bài] Đã tạo file submission.zip thành công!')
